In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
data_path = Path("../data/raw/solar/b7bc5a9a6b8db8501fc3907545fa2d4c.csv")

print("File exists:", data_path.exists())

File exists: True


In [3]:
df = pd.read_csv(
    data_path,
    sep=";",
    comment="#",
    header=None,
    names=[
        "observation_period",
        "toa",
        "clear_sky_ghi",
        "clear_sky_bhi",
        "clear_sky_dhi",
        "clear_sky_bni",
        "ghi",
        "bhi",
        "dhi",
        "bni",
        "reliability",
    ],
)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (744, 11)

Columns:
['observation_period', 'toa', 'clear_sky_ghi', 'clear_sky_bhi', 'clear_sky_dhi', 'clear_sky_bni', 'ghi', 'bhi', 'dhi', 'bni', 'reliability']


In [4]:
periods = df["observation_period"].str.split("/", expand=True)

df["start_time"] = pd.to_datetime(periods[0])
df["end_time"] = pd.to_datetime(periods[1])

In [5]:
df[["observation_period", "start_time", "end_time"]].head()

,observation_period,start_time,end_time
0,2020-01-01T00:00:00.0/2020-01-01T01:00:00.0,2020-01-01 00:00:00,2020-01-01 01:00:00
1,2020-01-01T01:00:00.0/2020-01-01T02:00:00.0,2020-01-01 01:00:00,2020-01-01 02:00:00
2,2020-01-01T02:00:00.0/2020-01-01T03:00:00.0,2020-01-01 02:00:00,2020-01-01 03:00:00
3,2020-01-01T03:00:00.0/2020-01-01T04:00:00.0,2020-01-01 03:00:00,2020-01-01 04:00:00
4,2020-01-01T04:00:00.0/2020-01-01T05:00:00.0,2020-01-01 04:00:00,2020-01-01 05:00:00


In [6]:
# ==========================================
# DATA QUALITY REPORT
# ==========================================

numeric_columns = [
    "toa",
    "clear_sky_ghi",
    "clear_sky_bhi",
    "clear_sky_dhi",
    "clear_sky_bni",
    "ghi",
    "bhi",
    "dhi",
    "bni",
    "reliability",
]

print("========== DATA QUALITY REPORT ==========")

# 1. Dataset dimensions
print("\n1. Dataset dimensions")
print("Rows:", len(df))
print("Columns:", len(df.columns))

# 2. Missing values
print("\n2. Missing values")
print(df[numeric_columns].isna().sum())

# 3. Duplicate observation periods
print("\n3. Duplicate observation periods")
print(df["observation_period"].duplicated().sum())

# 4. Duplicate timestamps
print("\n4. Duplicate start timestamps")
print(df["start_time"].duplicated().sum())

# 5. Interval duration
duration_hours = (
    (df["end_time"] - df["start_time"])
    .dt.total_seconds() / 3600
)

print("\n5. Observation duration")
print(duration_hours.value_counts().sort_index())

# 6. Temporal gaps / overlaps
time_difference = df["start_time"].diff()

unexpected_intervals = (
    time_difference.iloc[1:] != pd.Timedelta(hours=1)
).sum()

print("\n6. Unexpected temporal intervals")
print(unexpected_intervals)

# 7. Negative values
print("\n7. Negative values")

for column in numeric_columns:
    count = (df[column] < 0).sum()
    print(f"{column}: {count}")

# 8. Reliability range
print("\n8. Reliability range")
print("Minimum:", df["reliability"].min())
print("Maximum:", df["reliability"].max())

# 9. GHI > clear-sky GHI
print("\n9. GHI greater than clear-sky GHI")
print(
    (df["ghi"] > df["clear_sky_ghi"]).sum()
)

# 10. Nighttime consistency
night_mask = df["clear_sky_ghi"] == 0

print("\n10. Nighttime observations")
print("Nighttime rows:", night_mask.sum())
print(
    "Nighttime rows with positive GHI:",
    (df.loc[night_mask, "ghi"] > 0).sum()
)

print("\n==========================================")

========== DATA QUALITY REPORT ==========

1. Dataset dimensions
Rows: 744
Columns: 13

2. Missing values
toa              0
clear_sky_ghi    0
clear_sky_bhi    0
clear_sky_dhi    0
clear_sky_bni    0
ghi              0
bhi              0
dhi              0
bni              0
reliability      0
dtype: int64

3. Duplicate observation periods
0

4. Duplicate start timestamps
0

5. Observation duration
1.0    744
Name: count, dtype: int64

6. Unexpected temporal intervals
0

7. Negative values
toa: 0
clear_sky_ghi: 0
clear_sky_bhi: 0
clear_sky_dhi: 0
clear_sky_bni: 0
ghi: 0
bhi: 0
dhi: 0
bni: 0
reliability: 0

8. Reliability range
Minimum: 0.65
Maximum: 1.0

9. GHI greater than clear-sky GHI
0

10. Nighttime observations
Nighttime rows: 448
Nighttime rows with positive GHI: 0



In [7]:
print("Reliability summary:")
print(df["reliability"].describe())

print("\nReliability value counts:")
print(
    df["reliability"]
    .value_counts()
    .sort_index()
)

Reliability summary:
count    744.000000
mean       0.976086
std        0.076540
min        0.650000
25%        1.000000
50%        1.000000
75%        1.000000
max        1.000000
Name: reliability, dtype: float64

Reliability value counts:
reliability
0.6500      2
0.6583      2
0.6667      3
0.6750      3
0.6833      4
0.6917      7
0.7000      3
0.7083      1
0.7167      7
0.7250      2
0.7333      3
0.7417      1
0.7500      3
0.7583      2
0.7667      2
0.7750      1
0.7833     11
0.7917      1
0.8000      1
0.8083      1
0.8417      2
0.8583      1
0.8833      1
0.8917      1
0.9000      1
0.9167      1
0.9250      1
0.9417      2
0.9500      2
0.9583      1
0.9667      1
0.9750      1
0.9833      3
0.9917      1
1.0000    665
Name: count, dtype: int64


In [8]:
print(
    "Rows with reliability < 1:",
    (df["reliability"] < 1).sum()
)

print(
    "Rows with reliability = 1:",
    (df["reliability"] == 1).sum()
)

Rows with reliability < 1: 79
Rows with reliability = 1: 665


In [9]:
df["reliability_flag"] = np.where(
    df["reliability"] < 1.0,
    "reduced_reliability",
    "full_reliability"
)

In [10]:
df["reliability_flag"].value_counts()

reliability_flag
full_reliability       665
reduced_reliability     79
Name: count, dtype: int64

In [11]:
daylight_mask = df["clear_sky_ghi"] > 0

print("Daylight observations:", daylight_mask.sum())

print(
    "Observed GHI < 0:",
    (df.loc[daylight_mask, "ghi"] < 0).sum()
)

print(
    "Observed GHI > clear-sky GHI:",
    (
        df.loc[daylight_mask, "ghi"]
        > df.loc[daylight_mask, "clear_sky_ghi"]
    ).sum()
)

print(
    "Observed GHI = clear-sky GHI:",
    (
        df.loc[daylight_mask, "ghi"]
        == df.loc[daylight_mask, "clear_sky_ghi"]
    ).sum()
)

Daylight observations: 296
Observed GHI < 0: 0
Observed GHI > clear-sky GHI: 0
Observed GHI = clear-sky GHI: 175


In [12]:
# ==========================================
# RELIABILITY VS SOLAR CONDITIONS
# ==========================================

reduced = df[df["reliability"] < 1].copy()

print("Reduced-reliability observations:", len(reduced))

print("\nReliability statistics:")
print(reduced["reliability"].describe())

print("\nReduced-reliability observations by nighttime/daylight:")

reduced_daylight = (
    reduced["clear_sky_ghi"] > 0
).sum()

reduced_nighttime = (
    reduced["clear_sky_ghi"] == 0
).sum()

print("Daylight:", reduced_daylight)
print("Nighttime:", reduced_nighttime)

print("\nGHI statistics for reduced-reliability observations:")
print(reduced["ghi"].describe())

print("\nClear-sky GHI statistics for reduced-reliability observations:")
print(reduced["clear_sky_ghi"].describe())

print("\nReliability and timestamp examples:")
print(
    reduced[
        [
            "start_time",
            "ghi",
            "clear_sky_ghi",
            "reliability"
        ]
    ].head(20)
)

Reduced-reliability observations: 79

Reliability statistics:
count    79.000000
mean      0.774789
std       0.099444
min       0.650000
25%       0.691700
50%       0.750000
75%       0.804150
max       0.991700
Name: reliability, dtype: float64

Reduced-reliability observations by nighttime/daylight:
Daylight: 79
Nighttime: 0

GHI statistics for reduced-reliability observations:
count     79.000000
mean      31.137813
std       23.503892
min        0.005200
25%       13.314800
50%       30.352000
75%       44.424650
max      101.336900
Name: ghi, dtype: float64

Clear-sky GHI statistics for reduced-reliability observations:
count     79.000000
mean      37.976971
std       26.017792
min        0.008100
25%       19.683850
50%       37.345500
75%       55.397600
max      101.336900
Name: clear_sky_ghi, dtype: float64

Reliability and timestamp examples:
             start_time      ghi  clear_sky_ghi  reliability
7   2020-01-01 07:00:00  36.0358        36.0358       0.6917
15  2020-0

In [13]:
# ==========================================
# RELIABILITY PATTERN ANALYSIS
# ==========================================

# Create useful derived variables
df["hour"] = df["start_time"].dt.hour

df["daylight"] = df["clear_sky_ghi"] > 0

# Reliability groups
df["reliability_group"] = np.where(
    df["reliability"] < 1.0,
    "reduced",
    "full"
)

print("========== RELIABILITY PATTERN ANALYSIS ==========")

# --------------------------------------------------
# 1. Reliability by hour of day
# --------------------------------------------------

print("\n1. Reduced-reliability observations by hour:")

reduced_by_hour = (
    df[df["reliability"] < 1.0]
    .groupby("hour")
    .size()
)

print(reduced_by_hour)

# --------------------------------------------------
# 2. Compare solar conditions
# --------------------------------------------------

print("\n2. Solar conditions by reliability group:")

solar_comparison = (
    df[df["daylight"]]
    .groupby("reliability_group")[
        ["ghi", "clear_sky_ghi", "reliability"]
    ]
    .agg(["count", "mean", "median", "min", "max"])
)

print(solar_comparison)

# --------------------------------------------------
# 3. GHI comparison
# --------------------------------------------------

print("\n3. GHI statistics:")

print(
    df[df["daylight"]]
    .groupby("reliability_group")["ghi"]
    .describe()
)

# --------------------------------------------------
# 4. Clear-sky GHI comparison
# --------------------------------------------------

print("\n4. Clear-sky GHI statistics:")

print(
    df[df["daylight"]]
    .groupby("reliability_group")["clear_sky_ghi"]
    .describe()
)

# --------------------------------------------------
# 5. Reliability vs GHI correlation
# --------------------------------------------------

print("\n5. Correlation between reliability and GHI:")

print(
    df[df["daylight"]][
        ["reliability", "ghi", "clear_sky_ghi"]
    ].corr()
)

# --------------------------------------------------
# 6. Reduced reliability percentage by hour
# --------------------------------------------------

print("\n6. Percentage of observations with reduced reliability by hour:")

hour_summary = (
    df[df["daylight"]]
    .groupby("hour")
    .agg(
        observations=("reliability", "size"),
        reduced_reliability=(
            "reliability",
            lambda x: (x < 1.0).sum()
        )
    )
)

hour_summary["reduced_percentage"] = (
    hour_summary["reduced_reliability"]
    / hour_summary["observations"]
    * 100
)

print(hour_summary)

print("\n==============================================")

========== RELIABILITY PATTERN ANALYSIS ==========

1. Reduced-reliability observations by hour:
hour
6      6
7     31
15    31
16    11
dtype: int64

2. Solar conditions by reliability group:
                    ghi                                           \
                  count        mean    median      min       max   
reliability_group                                                  
full                217  245.172809  257.3925  26.4648  457.1864   
reduced              79   31.137813   30.3520   0.0052  101.3369   

                  clear_sky_ghi                                            \
                          count        mean    median       min       max   
reliability_group                                                           
full                        217  312.145387  332.0425  131.5380  479.1154   
reduced                      79   37.976971   37.3455    0.0081  101.3369   

                  reliability                                 
                

In [14]:
# ==========================================
# PHYSICAL CONSISTENCY CHECKS
# ==========================================

print("========== PHYSICAL CONSISTENCY CHECKS ==========")

# --------------------------------------------------
# 1. Negative values
# --------------------------------------------------

radiation_columns = [
    "toa",
    "clear_sky_ghi",
    "clear_sky_bhi",
    "clear_sky_dhi",
    "clear_sky_bni",
    "ghi",
    "bhi",
    "dhi",
    "bni",
]

print("\n1. Negative values:")

for column in radiation_columns:
    negative_count = (df[column] < 0).sum()
    print(f"{column}: {negative_count}")


# --------------------------------------------------
# 2. GHI vs BHI + DHI
# --------------------------------------------------

df["component_sum"] = df["bhi"] + df["dhi"]

df["ghi_component_difference"] = (
    df["ghi"] - df["component_sum"]
)

print("\n2. GHI - (BHI + DHI):")

print(
    df["ghi_component_difference"].describe()
)

print(
    "Absolute difference > 0.01:",
    (
        df["ghi_component_difference"].abs() > 0.01
    ).sum()
)


# --------------------------------------------------
# 3. Observed GHI vs clear-sky GHI
# --------------------------------------------------

print("\n3. GHI > clear-sky GHI:")

print(
    (df["ghi"] > df["clear_sky_ghi"]).sum()
)


# --------------------------------------------------
# 4. Observed BHI vs clear-sky BHI
# --------------------------------------------------

print("\n4. BHI > clear-sky BHI:")

print(
    (df["bhi"] > df["clear_sky_bhi"]).sum()
)


# --------------------------------------------------
# 5. Observed DHI vs clear-sky DHI
# --------------------------------------------------

print("\n5. DHI > clear-sky DHI:")

print(
    (df["dhi"] > df["clear_sky_dhi"]).sum()
)


# --------------------------------------------------
# 6. Observed BNI vs clear-sky BNI
# --------------------------------------------------

print("\n6. BNI > clear-sky BNI:")

print(
    (df["bni"] > df["clear_sky_bni"]).sum()
)


# --------------------------------------------------
# 7. Daylight-only component consistency
# --------------------------------------------------

daylight = df[df["clear_sky_ghi"] > 0].copy()

print("\n7. Daylight component statistics:")

print(
    daylight[
        [
            "ghi",
            "bhi",
            "dhi",
            "bni",
            "clear_sky_ghi",
            "clear_sky_bhi",
            "clear_sky_dhi",
            "clear_sky_bni"
        ]
    ].describe()
)


# --------------------------------------------------
# 8. Exact equality with clear-sky values
# --------------------------------------------------

print("\n8. Daylight observations where observed = clear-sky:")

for column in ["ghi", "bhi", "dhi", "bni"]:

    clear_column = "clear_sky_" + column

    equal_count = (
        daylight[column] == daylight[clear_column]
    ).sum()

    print(
        f"{column}: {equal_count} / {len(daylight)}"
    )


# --------------------------------------------------
# 9. Zero relationships
# --------------------------------------------------

print("\n9. Daylight zero counts:")

for column in ["ghi", "bhi", "dhi", "bni"]:

    zero_count = (daylight[column] == 0).sum()

    print(
        f"{column}: {zero_count}"
    )


print("\n==============================================")

========== PHYSICAL CONSISTENCY CHECKS ==========

1. Negative values:
toa: 0
clear_sky_ghi: 0
clear_sky_bhi: 0
clear_sky_dhi: 0
clear_sky_bni: 0
ghi: 0
bhi: 0
dhi: 0
bni: 0

2. GHI - (BHI + DHI):
count    744.000000
mean       0.000002
std        0.000031
min       -0.000100
25%        0.000000
50%        0.000000
75%        0.000000
max        0.000100
Name: ghi_component_difference, dtype: float64
Absolute difference > 0.01: 0

3. GHI > clear-sky GHI:
0

4. BHI > clear-sky BHI:
0

5. DHI > clear-sky DHI:
83

6. BNI > clear-sky BNI:
0

7. Daylight component statistics:
              ghi         bhi         dhi          bni  clear_sky_ghi  \
count  296.000000  296.000000  296.000000   296.000000     296.000000   
mean   188.048604  133.809351   54.239249   480.651775     238.972059   
std    140.569815  131.556278   38.749980   377.938139     146.155616   
min      0.005200    0.000000    0.005200     0.000000       0.008100   
25%     54.570875    2.460100   29.545625    13.410000   

In [15]:
# ==========================================
# INVESTIGATE BNI > CLEAR-SKY BNI
# ==========================================

bni_excess = df[
    df["bni"] > df["clear_sky_bni"]
].copy()

print("========== BNI EXCESS INVESTIGATION ==========")

print("\nNumber of observations:")
print(len(bni_excess))

print("\nBNI statistics:")
print(
    bni_excess[
        ["bni", "clear_sky_bni"]
    ].describe()
)

# Difference
bni_excess["bni_difference"] = (
    bni_excess["bni"] -
    bni_excess["clear_sky_bni"]
)

print("\nBNI - clear-sky BNI:")
print(
    bni_excess["bni_difference"].describe()
)

# Relative difference
bni_excess["bni_relative_difference"] = (
    bni_excess["bni_difference"] /
    bni_excess["clear_sky_bni"]
)

print("\nRelative difference:")
print(
    bni_excess["bni_relative_difference"].describe()
)

# Inspect timestamps and related radiation variables
print("\nDetailed observations:")

print(
    bni_excess[
        [
            "start_time",
            "toa",
            "ghi",
            "bhi",
            "dhi",
            "bni",
            "clear_sky_bni",
            "reliability"
        ]
    ].to_string(index=False)
)

print("\n==============================================")

========== BNI EXCESS INVESTIGATION ==========

Number of observations:
0

BNI statistics:
       bni  clear_sky_bni
count  0.0            0.0
mean   NaN            NaN
std    NaN            NaN
min    NaN            NaN
25%    NaN            NaN
50%    NaN            NaN
75%    NaN            NaN
max    NaN            NaN

BNI - clear-sky BNI:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: bni_difference, dtype: float64

Relative difference:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: bni_relative_difference, dtype: float64

Detailed observations:
Empty DataFrame
Columns: [start_time, toa, ghi, bhi, dhi, bni, clear_sky_bni, reliability]
Index: []



In [16]:
# ==========================================
# RADIATION COMPONENT ZERO-CONSISTENCY CHECK
# ==========================================

print("========== COMPONENT ZERO-CONSISTENCY ==========")

checks = {
    "GHI > 0 but BHI = 0":
        ((df["ghi"] > 0) & (df["bhi"] == 0)).sum(),

    "GHI > 0 but DHI = 0":
        ((df["ghi"] > 0) & (df["dhi"] == 0)).sum(),

    "GHI = 0 but BHI > 0":
        ((df["ghi"] == 0) & (df["bhi"] > 0)).sum(),

    "GHI = 0 but DHI > 0":
        ((df["ghi"] == 0) & (df["dhi"] > 0)).sum(),

    "GHI = 0 but BNI > 0":
        ((df["ghi"] == 0) & (df["bni"] > 0)).sum(),

    "BNI > 0 but BHI = 0":
        ((df["bni"] > 0) & (df["bhi"] == 0)).sum(),
}

for description, count in checks.items():
    print(f"{description}: {count}")

print("\n==============================================")

========== COMPONENT ZERO-CONSISTENCY ==========
GHI > 0 but BHI = 0: 23
GHI > 0 but DHI = 0: 0
GHI = 0 but BHI > 0: 0
GHI = 0 but DHI > 0: 0
GHI = 0 but BNI > 0: 0
BNI > 0 but BHI = 0: 4



In [17]:
# ==========================================
# INSPECT ZERO-COMPONENT OBSERVATIONS
# ==========================================

print("========== GHI > 0 BUT BHI = 0 ==========")

ghi_positive_bhi_zero = df[
    (df["ghi"] > 0) &
    (df["bhi"] == 0)
]

print(
    ghi_positive_bhi_zero[
        [
            "start_time",
            "toa",
            "ghi",
            "bhi",
            "dhi",
            "bni",
            "clear_sky_ghi",
            "reliability"
        ]
    ].to_string(index=False)
)

print("\nCount:", len(ghi_positive_bhi_zero))


print("\n========== BNI > 0 BUT BHI = 0 ==========")

bni_positive_bhi_zero = df[
    (df["bni"] > 0) &
    (df["bhi"] == 0)
]

print(
    bni_positive_bhi_zero[
        [
            "start_time",
            "toa",
            "ghi",
            "bhi",
            "dhi",
            "bni",
            "clear_sky_ghi",
            "clear_sky_bhi",
            "clear_sky_bni",
            "reliability"
        ]
    ].to_string(index=False)
)

print("\nCount:", len(bni_positive_bhi_zero))

print("\n==============================================")

========== GHI > 0 BUT BHI = 0 ==========
         start_time      toa     ghi  bhi     dhi    bni  clear_sky_ghi  reliability
2020-01-03 10:00:00 461.7180 68.3846  0.0 68.3846 0.0000       341.7896       1.0000
2020-01-04 12:00:00 454.5181 53.4293  0.0 53.4293 0.0000       339.7133       1.0000
2020-01-04 13:00:00 361.8391 43.5166  0.0 43.5166 0.0000       258.8030       1.0000
2020-01-08 15:00:00  54.2527  8.4261  0.0  8.4261 0.0000        26.4650       0.7917
2020-01-17 15:00:00  88.1387 15.2022  0.0 15.2022 0.0000        44.9979       0.6917
2020-01-18 08:00:00 271.8434 37.7054  0.0 37.7054 0.0000       183.6472       1.0000
2020-01-21 15:00:00 108.1483 22.4154  0.0 22.4154 0.0001        61.1070       0.7833
2020-01-21 16:00:00   0.0339  0.0052  0.0  0.0052 0.0000         0.0081       0.9833
2020-01-26 06:00:00   0.0787  0.0124  0.0  0.0124 0.0000         0.0192       0.9917
2020-01-26 08:00:00 302.8962 51.1801  0.0 51.1801 0.0001       200.1177       1.0000
2020-01-26 16:00:00   2

In [18]:
# ==========================================
# INSPECT THE 4 BNI > 0 BUT BHI = 0 CASES
# ==========================================

bni_positive_bhi_zero = df[
    (df["bni"] > 0) &
    (df["bhi"] == 0)
].copy()

print("Number of observations:", len(bni_positive_bhi_zero))

print(
    bni_positive_bhi_zero[
        [
            "start_time",
            "toa",
            "ghi",
            "bhi",
            "dhi",
            "bni",
            "clear_sky_ghi",
            "clear_sky_bhi",
            "clear_sky_bni",
            "reliability"
        ]
    ].to_string(index=False)
)

Number of observations: 4
         start_time      toa     ghi  bhi     dhi    bni  clear_sky_ghi  clear_sky_bhi  clear_sky_bni  reliability
2020-01-21 15:00:00 108.1483 22.4154  0.0 22.4154 0.0001        61.1070        41.2255       427.3700       0.7833
2020-01-26 08:00:00 302.8962 51.1801  0.0 51.1801 0.0001       200.1177       139.0975       634.9182       1.0000
2020-01-29 07:00:00 125.7838 30.1595  0.0 30.1595 0.0005        69.9193        40.8558       372.1057       0.7167
2020-01-29 09:00:00 464.9774 86.6734  0.0 86.6734 0.0001       353.3922       287.2804       865.4062       1.0000


In [19]:
# ==========================================
# FINAL QA FLAGS
# ==========================================

# Reliability flag
df["reliability_flag"] = np.where(
    df["reliability"] < 1.0,
    "reduced_reliability",
    "full_reliability"
)

# Component consistency flag
df["component_consistency_flag"] = np.where(
    df["ghi_component_difference"].abs() <= 0.01,
    "consistent",
    "check"
)

# Clear-sky comparison flag
df["clear_sky_consistency_flag"] = np.where(
    (
        (df["ghi"] <= df["clear_sky_ghi"]) &
        (df["bhi"] <= df["clear_sky_bhi"]) &
        (df["dhi"] <= df["clear_sky_dhi"]) &
        (df["bni"] <= df["clear_sky_bni"])
    ),
    "consistent",
    "check"
)

# Nighttime flag
df["solar_period_flag"] = np.where(
    df["clear_sky_ghi"] == 0,
    "nighttime",
    "daylight"
)

print("========== FINAL QA FLAG SUMMARY ==========")

print("\nReliability:")
print(df["reliability_flag"].value_counts())

print("\nComponent consistency:")
print(df["component_consistency_flag"].value_counts())

print("\nClear-sky consistency:")
print(df["clear_sky_consistency_flag"].value_counts())

print("\nSolar period:")
print(df["solar_period_flag"].value_counts())

print("\nRows requiring component investigation:")
print(
    (
        df["component_consistency_flag"] == "check"
    ).sum()
)

print("\nRows requiring clear-sky investigation:")
print(
    (
        df["clear_sky_consistency_flag"] == "check"
    ).sum()
)

print("\n==============================================")

========== FINAL QA FLAG SUMMARY ==========

Reliability:
reliability_flag
full_reliability       665
reduced_reliability     79
Name: count, dtype: int64

Component consistency:
component_consistency_flag
consistent    744
Name: count, dtype: int64

Clear-sky consistency:
clear_sky_consistency_flag
consistent    661
check          83
Name: count, dtype: int64

Solar period:
solar_period_flag
nighttime    448
daylight     296
Name: count, dtype: int64

Rows requiring component investigation:
0

Rows requiring clear-sky investigation:
83



In [20]:
# ==========================================
# DIAGNOSE CLEAR-SKY CONSISTENCY FLAGS
# ==========================================

print("========== CLEAR-SKY FLAG DIAGNOSTIC ==========")

# Individual exceedance counts
print("\n1. Individual exceedance counts:")

print(
    "GHI > clear-sky GHI:",
    (df["ghi"] > df["clear_sky_ghi"]).sum()
)

print(
    "BHI > clear-sky BHI:",
    (df["bhi"] > df["clear_sky_bhi"]).sum()
)

print(
    "DHI > clear-sky DHI:",
    (df["dhi"] > df["clear_sky_dhi"]).sum()
)

print(
    "BNI > clear-sky BNI:",
    (df["bni"] > df["clear_sky_bni"]).sum()
)


# --------------------------------------------------
# Check for missing values in these columns
# --------------------------------------------------

print("\n2. Missing values:")

clearsky_columns = [
    "ghi",
    "clear_sky_ghi",
    "bhi",
    "clear_sky_bhi",
    "dhi",
    "clear_sky_dhi",
    "bni",
    "clear_sky_bni"
]

print(
    df[clearsky_columns].isna().sum()
)


# --------------------------------------------------
# Identify exactly which rows were flagged
# --------------------------------------------------

check_rows = df[
    df["clear_sky_consistency_flag"] == "check"
].copy()

print("\n3. Number of flagged rows:")
print(len(check_rows))


# --------------------------------------------------
# Show the actual values
# --------------------------------------------------

print("\n4. Flagged observations:")

print(
    check_rows[
        [
            "start_time",
            "ghi",
            "clear_sky_ghi",
            "bhi",
            "clear_sky_bhi",
            "dhi",
            "clear_sky_dhi",
            "bni",
            "clear_sky_bni",
            "reliability"
        ]
    ].to_string(index=False)
)


# --------------------------------------------------
# Determine which component caused each flag
# --------------------------------------------------

print("\n5. Component causing the flag:")

for column in ["ghi", "bhi", "dhi", "bni"]:

    clear_column = "clear_sky_" + column

    count = (
        check_rows[column] >
        check_rows[clear_column]
    ).sum()

    print(
        f"{column} > {clear_column}: {count}"
    )


print("\n==============================================")

========== CLEAR-SKY FLAG DIAGNOSTIC ==========

1. Individual exceedance counts:
GHI > clear-sky GHI: 0
BHI > clear-sky BHI: 0
DHI > clear-sky DHI: 83
BNI > clear-sky BNI: 0

2. Missing values:
ghi              0
clear_sky_ghi    0
bhi              0
clear_sky_bhi    0
dhi              0
clear_sky_dhi    0
bni              0
clear_sky_bni    0
dtype: int64

3. Number of flagged rows:
83

4. Flagged observations:
         start_time      ghi  clear_sky_ghi      bhi  clear_sky_bhi      dhi  clear_sky_dhi      bni  clear_sky_bni  reliability
2020-01-03 07:00:00  26.0812        31.0171   1.2688        15.7834  24.8124        15.2337  16.4960       203.6194       0.6917
2020-01-03 08:00:00 106.0336       154.6064   3.3870       108.8218 102.6466        45.7846  19.3391       621.7761       1.0000
2020-01-03 09:00:00  60.9950       269.1910   0.0309       208.2626  60.9640        60.9285   0.1335       773.5474       1.0000
2020-01-03 11:00:00  75.0062       363.3470   0.0017       290.6432

In [21]:
# ==========================================
# CLEAR-SKY DEVIATION FLAGS
# ==========================================

df["ghi_above_clear"] = (
    df["ghi"] > df["clear_sky_ghi"]
)

df["bhi_above_clear"] = (
    df["bhi"] > df["clear_sky_bhi"]
)

df["dhi_above_clear"] = (
    df["dhi"] > df["clear_sky_dhi"]
)

df["bni_above_clear"] = (
    df["bni"] > df["clear_sky_bni"]
)

print("========== CLEAR-SKY DEVIATION SUMMARY ==========")

print(
    "GHI above clear-sky:",
    df["ghi_above_clear"].sum()
)

print(
    "BHI above clear-sky:",
    df["bhi_above_clear"].sum()
)

print(
    "DHI above clear-sky:",
    df["dhi_above_clear"].sum()
)

print(
    "BNI above clear-sky:",
    df["bni_above_clear"].sum()
)

print("\nRows with any component above clear-sky:")

any_above = (
    df[
        [
            "ghi_above_clear",
            "bhi_above_clear",
            "dhi_above_clear",
            "bni_above_clear"
        ]
    ].any(axis=1)
)

print(any_above.sum())

print("\n==============================================")

========== CLEAR-SKY DEVIATION SUMMARY ==========
GHI above clear-sky: 0
BHI above clear-sky: 0
DHI above clear-sky: 83
BNI above clear-sky: 0

Rows with any component above clear-sky:
83



In [25]:
# ==========================================
# FINAL EXTREME / TEMPORAL QA
# ==========================================

print("========== EXTREME / TEMPORAL QA ==========")

# --------------------------------------------------
# 1. Daylight observations
# --------------------------------------------------

daylight = df[df["clear_sky_ghi"] > 0].copy()

daylight["csi"] = (
    daylight["ghi"] / daylight["clear_sky_ghi"]
)

print("\n1. Daylight observations:")
print(len(daylight))


# --------------------------------------------------
# 2. CSI statistics
# --------------------------------------------------

print("\n2. CSI statistics:")

print(
    daylight["csi"].describe(
        percentiles=[
            0.01,
            0.05,
            0.10,
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99
        ]
    )
)


# --------------------------------------------------
# 3. Very low CSI observations
# --------------------------------------------------

print("\n3. Very low CSI observations:")

for threshold in [0.1, 0.2, 0.3]:

    count = (
        daylight["csi"] < threshold
    ).sum()

    print(
        f"CSI < {threshold}: {count}"
    )


# --------------------------------------------------
# 4. Highest GHI observations
# --------------------------------------------------

print("\n4. Highest GHI observations:")

highest_ghi = (
    daylight[
        [
            "start_time",
            "ghi",
            "clear_sky_ghi",
            "csi",
            "reliability"
        ]
    ]
    .sort_values(
        "ghi",
        ascending=False
    )
    .head(10)
)

print(
    highest_ghi.to_string(index=False)
)


# --------------------------------------------------
# 5. Lowest CSI observations
# --------------------------------------------------

print("\n5. Lowest CSI observations:")

lowest_csi = (
    daylight[
        [
            "start_time",
            "ghi",
            "clear_sky_ghi",
            "csi",
            "reliability"
        ]
    ]
    .sort_values(
        "csi",
        ascending=True
    )
    .head(20)
)

print(
    lowest_csi.to_string(index=False)
)


# --------------------------------------------------
# 6. Hour-to-hour GHI changes
# --------------------------------------------------

df_sorted = (
    df
    .sort_values("start_time")
    .copy()
)

df_sorted["ghi_change"] = (
    df_sorted["ghi"].diff()
)

df_sorted["absolute_ghi_change"] = (
    df_sorted["ghi_change"].abs()
)

print("\n6. Absolute hour-to-hour GHI change:")

print(
    df_sorted["absolute_ghi_change"].describe()
)


# --------------------------------------------------
# 7. Largest hour-to-hour GHI changes
# --------------------------------------------------

print("\n7. Largest hour-to-hour GHI changes:")

largest_changes = (
    df_sorted[
        [
            "start_time",
            "ghi",
            "ghi_change",
            "absolute_ghi_change",
            "reliability"
        ]
    ]
    .sort_values(
        "absolute_ghi_change",
        ascending=False
    )
    .head(20)
)

print(
    largest_changes.to_string(index=False)
)


# --------------------------------------------------
# 8. Very low CSI + reliability
# --------------------------------------------------

print("\n8. CSI < 0.2 with reliability:")

low_csi = daylight[
    daylight["csi"] < 0.2
].copy()

print(
    low_csi[
        [
            "start_time",
            "ghi",
            "clear_sky_ghi",
            "csi",
            "reliability"
        ]
    ].to_string(index=False)
)

print(
    "\nNumber of CSI < 0.2 observations:",
    len(low_csi)
)


# --------------------------------------------------
# 9. Low CSI reliability summary
# --------------------------------------------------

print("\n9. Reliability summary for CSI < 0.2:")

if len(low_csi) > 0:

    print(
        low_csi["reliability"].describe()
    )

    print(
        "\nReliability < 1.0:",
        (low_csi["reliability"] < 1.0).sum()
    )

    print(
        "Reliability = 1.0:",
        (low_csi["reliability"] == 1.0).sum()
    )

else:

    print("No observations with CSI < 0.2")


print("\n==============================================")

========== EXTREME / TEMPORAL QA ==========

1. Daylight observations:
296

2. CSI statistics:
count    296.000000
mean       0.803429
std        0.293146
min        0.111461
1%         0.137729
5%         0.175853
10%        0.263017
25%        0.644869
50%        1.000000
75%        1.000000
90%        1.000000
95%        1.000000
99%        1.000000
max        1.000000
Name: csi, dtype: float64

3. Very low CSI observations:
CSI < 0.1: 0
CSI < 0.2: 18
CSI < 0.3: 36

4. Highest GHI observations:
         start_time      ghi  clear_sky_ghi      csi  reliability
2020-01-31 11:00:00 457.1864       476.7720 0.958920          1.0
2020-01-27 11:00:00 455.2359       455.2359 1.000000          1.0
2020-01-23 11:00:00 449.6596       449.6596 1.000000          1.0
2020-01-20 11:00:00 448.0645       448.0645 1.000000          1.0
2020-01-30 10:00:00 447.5039       447.5039 1.000000          1.0
2020-01-22 11:00:00 443.4452       443.4452 1.000000          1.0
2020-01-21 11:00:00 435.8155       

In [26]:
# ==========================================
# REMAINING EXTREME / TEMPORAL QA
# ==========================================

print("========== REMAINING EXTREME QA ==========")


# --------------------------------------------------
# 1. Lowest CSI observations
# --------------------------------------------------

print("\n1. 20 LOWEST CSI OBSERVATIONS:")

print(
    daylight[
        [
            "start_time",
            "ghi",
            "clear_sky_ghi",
            "csi",
            "reliability"
        ]
    ]
    .sort_values("csi")
    .head(20)
    .to_string(index=False)
)


# --------------------------------------------------
# 2. Largest hour-to-hour GHI changes
# --------------------------------------------------

print("\n2. 20 LARGEST HOUR-TO-HOUR GHI CHANGES:")

print(
    df_sorted[
        [
            "start_time",
            "ghi",
            "ghi_change",
            "absolute_ghi_change",
            "reliability"
        ]
    ]
    .sort_values(
        "absolute_ghi_change",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)


# --------------------------------------------------
# 3. Extreme-change summary
# --------------------------------------------------

print("\n3. ABSOLUTE GHI CHANGE SUMMARY:")

print(
    df_sorted["absolute_ghi_change"].describe(
        percentiles=[
            0.50,
            0.90,
            0.95,
            0.99
        ]
    )
)


print("\n==========================================")

========== REMAINING EXTREME QA ==========

1. 20 LOWEST CSI OBSERVATIONS:
         start_time     ghi  clear_sky_ghi      csi  reliability
2020-01-29 12:00:00 49.7493       446.3374 0.111461       1.0000
2020-01-28 12:00:00 52.8575       433.4192 0.121955       1.0000
2020-01-28 11:00:00 59.8738       458.3466 0.130630       1.0000
2020-01-29 08:00:00 30.7736       222.8318 0.138102       1.0000
2020-01-29 11:00:00 66.6011       470.9982 0.141404       1.0000
2020-01-29 16:00:00  0.2625         1.7286 0.151857       0.8917
2020-01-28 16:00:00  0.1807         1.1547 0.156491       0.9000
2020-01-04 12:00:00 53.4293       339.7133 0.157278       1.0000
2020-01-29 10:00:00 70.6865       438.9877 0.161022       1.0000
2020-01-26 16:00:00  0.0997         0.6162 0.161798       0.9250
2020-01-19 07:00:00  8.1667        49.8969 0.163671       0.7583
2020-01-29 13:00:00 61.3504       367.2829 0.167039       1.0000
2020-01-04 13:00:00 43.5166       258.8030 0.168146       1.0000
2020-01-28 15:0

# Final Data Quality Assessment

## Dataset assessed

The January 2020 CAMS solar radiation time series for the Davos study location
(46.80° N, 9.83° E; 1610 m) contains 744 hourly observations.

## Data integrity

The dataset contains:

- 744 hourly observations
- No missing values in the assessed radiation and reliability variables
- No duplicate observation periods
- No duplicate start timestamps
- Continuous one-hour temporal intervals
- One-hour observation duration throughout the dataset

## Physical consistency

The radiation components satisfy the expected numerical relationship:

GHI ≈ BHI + DHI

with absolute differences below 0.01 Wh/m² for all observations.

No negative values were identified in the assessed radiation variables.

No positive GHI observations occurred when clear-sky GHI was zero.

No observed GHI, BHI, or BNI values exceeded their corresponding
clear-sky values.

Observed DHI exceeded clear-sky DHI in 83 observations. These observations
were retained because exceeding the clear-sky diffuse component does not,
by itself, establish that the observation is erroneous. Cloud and aerosol
conditions can alter the partition between beam and diffuse radiation.

## Reliability

The CAMS reliability variable ranges from 0.65 to 1.00.

- 665 observations have reliability = 1.00.
- 79 observations have reliability < 1.00.

All 79 reduced-reliability observations occur during daylight in this
January 2020 pilot.

Reduced-reliability observations are retained rather than automatically
removed because excluding them could preferentially remove observations
associated with low solar availability.

Reliability will therefore be retained as a quality indicator for subsequent
analysis.

## Extreme solar availability

The daylight clear-sky index (CSI), defined for exploratory purposes as

CSI = GHI / Clear-sky GHI

has the following characteristics:

- Mean: 0.8034
- Standard deviation: 0.2931
- Minimum: 0.1115
- 18 observations have CSI < 0.2
- 36 observations have CSI < 0.3

Of the 18 observations with CSI < 0.2:

- 13 have reliability = 1.00
- 5 have reliability < 1.00

Therefore, very low CSI values cannot be attributed solely to reduced
reliability.

Extreme low-CSI observations are retained because unusually low solar
availability is itself relevant to the research objective.

## Temporal extremes

The maximum absolute hour-to-hour GHI change observed in the pilot dataset
was 281.6942 Wh/m².

Large temporal changes are treated as diagnostic observations rather than
automatically classified as errors because rapid changes in solar radiation
may represent genuine atmospheric variability.

## Outlier handling policy

No observations are removed solely because they are statistically extreme,
have low CSI, or exhibit a large temporal change.

Potential quality issues will be handled using explicit quality flags and
documented criteria rather than arbitrary outlier deletion.

This approach is particularly important because extreme low-solar
observations may represent the physical phenomenon under investigation.

## Conclusion

The January 2020 pilot dataset passes the initial structural and physical
quality checks and is suitable for proceeding to solar normalization.

The pilot analysis does not establish drought events. Drought identification
requires a longer historical dataset and an explicitly defined climatological
reference, threshold, and persistence criterion.